
# 04.2 — Synthetic Black–Scholes Regression Baseline vs. 10-Run PINN

## Goal

Apply the **same nonlinear Black–Scholes regression calibration used for the real-data baseline** to the original synthetic European-call dataset, then compare the regression result with the **10-run synthetic inverse-PINN aggregate** from Notebook 04 Updated.

In [11]:

from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares
from scipy.special import ndtr

import inverse_pinn_training



## 1. Load the exact synthetic setting used by the PINN


In [12]:

K = float(inverse_pinn_training.K)
T = float(inverse_pinn_training.T)

t_min = float(inverse_pinn_training.t_min)
t_max = float(inverse_pinn_training.t_max)

S_min = float(inverse_pinn_training.S_min)
S_max = float(inverse_pinn_training.S_max)

r_true = float(inverse_pinn_training.r_true)
sigma_true = float(inverse_pinn_training.sigma_true)

N_f_pinn = int(inverse_pinn_training.N_f)

# Same 2,000 synthetic observations used by the PINN
t_obs_t, S_obs_t, V_obs_t = inverse_pinn_training.load_synthetic_data()

t_obs = t_obs_t.detach().cpu().numpy().reshape(-1)
S_obs = S_obs_t.detach().cpu().numpy().reshape(-1)
V_obs = V_obs_t.detach().cpu().numpy().reshape(-1)

N_obs = len(V_obs)

design_summary = pd.DataFrame(
    [
        ["Synthetic observations used for regression", N_obs],
        ["PINN PDE collocation points per run", N_f_pinn],
        ["PINN collocation structure", "2000 global"],
        ["Regression collocation points", "N/A — analytical BS calibration"],
        ["K", K],
        ["T", T],
        ["S_min", S_min],
        ["S_max", S_max],
        ["r_true", r_true],
        ["sigma_true", sigma_true],
    ],
    columns=["Setting", "Value"]
)

display(design_summary)

print(f"Loaded observations: {N_obs}")
print(f"t range: [{t_obs.min():.6f}, {t_obs.max():.6f}]")
print(f"S range: [{S_obs.min():.6f}, {S_obs.max():.6f}]")
print(f"V range: [{V_obs.min():.6f}, {V_obs.max():.6f}]")


,Setting,Value
0,Synthetic observations used for regression,2000
1,PINN PDE collocation points per run,2000
2,PINN collocation structure,2000 global
3,Regression collocation points,N/A — analytical BS calibration
4,K,40.0
5,T,1.0
6,S_min,0.0
7,S_max,160.0
8,r_true,0.05
9,sigma_true,0.2


Loaded observations: 2000
t range: [0.000641, 0.999085]
S range: [0.003953, 159.907040]
V range: [-0.000000, 121.744150]



## 2. Black–Scholes Eq. (5) in NumPy


In [13]:

def black_scholes_call_numpy(t, S, K, T, r, sigma):
    t = np.asarray(t, dtype=float)
    S = np.asarray(S, dtype=float)

    tau = T - t
    eps = 1e-12

    tau_safe = np.maximum(tau, eps)
    S_safe = np.maximum(S, eps)

    d1 = (
        np.log(S_safe / K)
        + (r + 0.5 * sigma**2) * tau_safe
    ) / (
        sigma * np.sqrt(tau_safe)
    )

    d2 = d1 - sigma * np.sqrt(tau_safe)

    V = (
        S_safe * ndtr(d1)
        - K * np.exp(-r * tau_safe) * ndtr(d2)
    )

    payoff = np.maximum(S - K, 0.0)

    V = np.where(tau <= eps, payoff, V)
    V = np.where(S <= eps, 0.0, V)

    return V



## 3. Fit the regression baseline on all 2,000 synthetic observations


In [14]:

x0 = np.array([0.10, 0.40], dtype=float)

lower_bounds = np.array([1e-8, 1e-8], dtype=float)
upper_bounds = np.array([1.0, 3.0], dtype=float)

def regression_residuals(theta):
    r, sigma = theta

    V_pred = black_scholes_call_numpy(
        t=t_obs,
        S=S_obs,
        K=K,
        T=T,
        r=r,
        sigma=sigma
    )

    return V_pred - V_obs


start_time = time.time()

fit = least_squares(
    regression_residuals,
    x0=x0,
    bounds=(lower_bounds, upper_bounds),
    method="trf",
    xtol=1e-14,
    ftol=1e-14,
    gtol=1e-14,
    max_nfev=10000
)

runtime_seconds = time.time() - start_time

r_reg = float(fit.x[0])
sigma_reg = float(fit.x[1])

V_reg_train = black_scholes_call_numpy(
    t=t_obs,
    S=S_obs,
    K=K,
    T=T,
    r=r_reg,
    sigma=sigma_reg
)

train_errors = V_reg_train - V_obs

train_mse = float(np.mean(train_errors**2))
train_rmse = float(np.sqrt(train_mse))
train_mae = float(np.mean(np.abs(train_errors)))
train_correlation = float(np.corrcoef(V_obs, V_reg_train)[0, 1])

r_absolute_error = abs(r_reg - r_true)
sigma_absolute_error = abs(sigma_reg - sigma_true)

r_error_percent = r_absolute_error / abs(r_true) * 100.0
sigma_error_percent = sigma_absolute_error / abs(sigma_true) * 100.0

print("Regression calibration completed")
print(f"function evaluations: {fit.nfev}")
print()
print(f"Estimated r: {r_reg:.12f}")
print(f"True r: {r_true:.12f}")
print(f"r absolute error: {r_absolute_error:.12e}")
print(f"r relative error: {r_error_percent:.8f}%")
print()
print(f"Estimated sigma: {sigma_reg:.12f}")
print(f"True sigma: {sigma_true:.12f}")
print(f"sigma absolute error: {sigma_absolute_error:.12e}")
print(f"sigma relative error: {sigma_error_percent:.8f}%")
print()
print(f"Training MSE (2000 obs): {train_mse:.12e}")
print(f"Training RMSE: {train_rmse:.12e}")
print(f"Training MAE: {train_mae:.12e}")
print(f"Training correlation: {train_correlation:.12f}")
print(f"Runtime: {runtime_seconds:.6f} seconds")


Regression calibration completed
success: True
message: `xtol` termination condition is satisfied.
function evaluations: 8

Estimated r: 0.050000004400
True r: 0.050000000000
r absolute error: 4.400000259974e-09
r relative error: 0.00000880%

Estimated sigma: 0.200000013003
True sigma: 0.200000000000
sigma absolute error: 1.300304305896e-08
sigma relative error: 0.00000650%

Training MSE (2000 obs): 1.206240223689e-11
Training RMSE: 3.473096923049e-06
Training MAE: 2.515164015992e-06
Training correlation: 1.000000000000
Runtime: 0.004869 seconds



## 4. Evaluate regression on the same fixed 100 × 100 test mesh as Notebook 04 Updated

This makes the regression price-surface MSE directly comparable to the PINN's reported test-surface MSE.


In [15]:

N_test = 100

t_test_values = np.linspace(t_min, t_max, N_test)
S_test_values = np.linspace(S_min, S_max, N_test)

t_mesh, S_mesh = np.meshgrid(
    t_test_values,
    S_test_values,
    indexing="ij"
)

t_test = t_mesh.reshape(-1)
S_test = S_mesh.reshape(-1)

V_true_test = black_scholes_call_numpy(
    t=t_test,
    S=S_test,
    K=K,
    T=T,
    r=r_true,
    sigma=sigma_true
)

V_reg_test = black_scholes_call_numpy(
    t=t_test,
    S=S_test,
    K=K,
    T=T,
    r=r_reg,
    sigma=sigma_reg
)

test_error = V_reg_test - V_true_test

test_mse = float(np.mean(test_error**2))
test_rmse = float(np.sqrt(test_mse))
test_mae = float(np.mean(np.abs(test_error)))
test_max_abs_error = float(np.max(np.abs(test_error)))
test_correlation = float(np.corrcoef(V_true_test, V_reg_test)[0, 1])

ss_res = float(np.sum((V_true_test - V_reg_test)**2))
ss_tot = float(np.sum((V_true_test - V_true_test.mean())**2))
test_r2 = 1.0 - ss_res / ss_tot

print("Fixed 100 x 100 test-mesh metrics")
print(f"Test MSE:                {test_mse:.12e}")
print(f"Test RMSE:               {test_rmse:.12e}")
print(f"Test MAE:                {test_mae:.12e}")
print(f"Test max absolute error: {test_max_abs_error:.12e}")
print(f"Test correlation:        {test_correlation:.12f}")
print(f"Test R^2:                {test_r2:.12f}")


Fixed 100 x 100 test-mesh metrics
Test MSE:                9.907749025733e-15
Test RMSE:               9.953767641317e-08
Test MAE:                7.612425354815e-08
Test max absolute error: 2.919716060035e-07
Test correlation:        1.000000000000
Test R^2:                1.000000000000



## 5. Save the regression results


In [16]:

regression_result = pd.DataFrame(
    [
        {
            "method": "Black-Scholes nonlinear least squares",
            "N_observations": N_obs,
            "regression_collocation_points": "N/A",
            "pinn_reference_collocation": "2000 global",
            "r_true": r_true,
            "sigma_true": sigma_true,
            "r_estimate": r_reg,
            "sigma_estimate": sigma_reg,
            "r_absolute_error": r_absolute_error,
            "sigma_absolute_error": sigma_absolute_error,
            "r_error_percent": r_error_percent,
            "sigma_error_percent": sigma_error_percent,
            "train_mse_2000_obs": train_mse,
            "train_rmse_2000_obs": train_rmse,
            "train_mae_2000_obs": train_mae,
            "train_correlation_2000_obs": train_correlation,
            "test_mesh_N": N_test * N_test,
            "test_mse": test_mse,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_max_abs_error": test_max_abs_error,
            "test_correlation": test_correlation,
            "test_r2": test_r2,
            "runtime_seconds": runtime_seconds,
            "optimizer_success": fit.success,
            "optimizer_nfev": fit.nfev,
        }
    ]
)

regression_result.to_csv(
    "04_2_synthetic_regression_result.csv",
    index=False
)

regression_test_predictions = pd.DataFrame(
    {
        "t": t_test,
        "S": S_test,
        "V_true": V_true_test,
        "V_regression": V_reg_test,
        "error_regression_minus_true": test_error,
    }
)

regression_test_predictions.to_csv(
    "04_2_synthetic_regression_test_predictions.csv",
    index=False
)

display(regression_result)

print("Saved:")
print("  04_2_synthetic_regression_result.csv")
print("  04_2_synthetic_regression_test_predictions.csv")


,method,N_observations,regression_collocation_points,pinn_reference_collocation,r_true,sigma_true,r_estimate,sigma_estimate,r_absolute_error,sigma_absolute_error,...,test_mesh_N,test_mse,test_rmse,test_mae,test_max_abs_error,test_correlation,test_r2,runtime_seconds,optimizer_success,optimizer_nfev
0,Black-Scholes nonlinear least squares,2000,N/A,2000 global,0.05,0.2,0.05,0.2,4.400000e-09,1.300304e-08,...,10000,9.907749e-15,9.953768e-08,7.612425e-08,2.919716e-07,1.0,1.0,0.004869,True,8


Saved:
  04_2_synthetic_regression_result.csv
  04_2_synthetic_regression_test_predictions.csv



## 6. Load the 10-run synthetic PINN results from Notebook 04 Updated

Expected files in the same `PINN` folder:

- `european_10_run_results.csv`
- `european_10_run_average_prediction_summary.csv`
- `european_10_run_average_prediction_test.csv`

The comparison uses the **10-run averaged PINN prediction** as the main PINN prediction metric.


In [17]:

pinn_runs_path = Path("european_10_run_results.csv")
pinn_summary_path = Path("european_10_run_average_prediction_summary.csv")
pinn_test_path = Path("european_10_run_average_prediction_test.csv")

pinn_runs_df = None
pinn_summary_df = None
pinn_test_df = None

if pinn_runs_path.exists():
    pinn_runs_df = pd.read_csv(pinn_runs_path)
    print(f"Loaded: {pinn_runs_path}")
else:
    print(f"Missing: {pinn_runs_path}")

if pinn_summary_path.exists():
    pinn_summary_df = pd.read_csv(pinn_summary_path)
    print(f"Loaded: {pinn_summary_path}")
else:
    print(f"Missing: {pinn_summary_path}")

if pinn_test_path.exists():
    pinn_test_df = pd.read_csv(pinn_test_path)
    print(f"Loaded: {pinn_test_path}")
else:
    print(f"Missing: {pinn_test_path}")


Loaded: european_10_run_results.csv
Missing: european_10_run_average_prediction_summary.csv
Missing: european_10_run_average_prediction_test.csv



## 7. Direct Regression vs. PINN comparison


In [18]:

comparison_rows = [
    {
        "method": "Regression",
        "runs": 1,
        "N_training_observations": N_obs,
        "PDE_collocation": "N/A",
        "r_estimate": r_reg,
        "r_std": 0.0,
        "sigma_estimate": sigma_reg,
        "sigma_std": 0.0,
        "r_error_percent": r_error_percent,
        "sigma_error_percent": sigma_error_percent,
        "test_mse_main": test_mse,
        "test_rmse_main": test_rmse,
        "test_mae_main": test_mae,
        "test_correlation_main": test_correlation,
        "test_r2_main": test_r2,
        "runtime_minutes_mean": runtime_seconds / 60.0,
    }
]

if (
    pinn_runs_df is not None
    and pinn_summary_df is not None
):
    ps = pinn_summary_df.iloc[0]

    pinn_main_mse = float(ps["pde_mse_average_prediction"])
    pinn_main_rmse = float(np.sqrt(pinn_main_mse))
    pinn_main_corr = float(ps["pde_correlation_average_prediction"])

    pinn_main_mae = np.nan
    pinn_main_r2 = np.nan

    if pinn_test_df is not None:
        required_cols = {
            "t",
            "S",
            "V_true",
            "V_pinn_average_10_runs"
        }

        if required_cols.issubset(pinn_test_df.columns):
            Vp_true = pinn_test_df["V_true"].to_numpy(dtype=float)
            Vp_pred = pinn_test_df["V_pinn_average_10_runs"].to_numpy(dtype=float)

            pinn_main_mae = float(
                np.mean(np.abs(Vp_pred - Vp_true))
            )

            ss_res_pinn = float(
                np.sum((Vp_true - Vp_pred)**2)
            )

            ss_tot_pinn = float(
                np.sum((Vp_true - Vp_true.mean())**2)
            )

            pinn_main_r2 = (
                1.0 - ss_res_pinn / ss_tot_pinn
            )

    comparison_rows.append(
        {
            "method": "PINN (10-run aggregate)",
            "runs": int(ps.get("N_runs", 10)),
            "N_training_observations": N_obs,
            "PDE_collocation": "2000 global per run",
            "r_estimate": float(ps["r_estimate_mean"]),
            "r_std": float(ps["r_estimate_std"]),
            "sigma_estimate": float(ps["sigma_estimate_mean"]),
            "sigma_std": float(ps["sigma_estimate_std"]),
            "r_error_percent": float(ps["r_error_percent_mean"]),
            "sigma_error_percent": float(ps["sigma_error_percent_mean"]),
            "test_mse_main": pinn_main_mse,
            "test_rmse_main": pinn_main_rmse,
            "test_mae_main": pinn_main_mae,
            "test_correlation_main": pinn_main_corr,
            "test_r2_main": pinn_main_r2,
            "runtime_minutes_mean": float(
                pinn_runs_df["runtime_minutes"].mean()
            )
            if "runtime_minutes" in pinn_runs_df.columns
            else np.nan,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

comparison_df.to_csv(
    "04_2_synthetic_regression_vs_pinn_summary.csv",
    index=False
)

print("Saved: 04_2_synthetic_regression_vs_pinn_summary.csv")


,method,runs,N_training_observations,PDE_collocation,r_estimate,r_std,sigma_estimate,sigma_std,r_error_percent,sigma_error_percent,test_mse_main,test_rmse_main,test_mae_main,test_correlation_main,test_r2_main,runtime_minutes_mean
0,Regression,1,2000,N/A,0.05,0.0,0.2,0.0,0.000009,0.000007,9.907749e-15,9.953768e-08,7.612425e-08,1.0,1.0,0.000081


Saved: 04_2_synthetic_regression_vs_pinn_summary.csv



## 8. PINN 10-run individual variability summary


In [19]:

if pinn_runs_df is not None:
    available_cols = [
        c for c in [
            "r_estimate",
            "sigma_estimate",
            "r_error_percent",
            "sigma_error_percent",
            "pde_mse",
            "pde_correlation",
            "runtime_minutes",
        ]
        if c in pinn_runs_df.columns
    ]

    if available_cols:
        pinn_individual_summary = (
            pinn_runs_df[available_cols]
            .agg(["mean", "std", "min", "max"])
        )

        display(pinn_individual_summary)

        pinn_individual_summary.to_csv(
            "04_2_pinn_10_run_individual_summary.csv"
        )

        print("Saved: 04_2_pinn_10_run_individual_summary.csv")
else:
    print(
        "PINN individual-run file not found. "
        "Regression results above are still valid."
    )


,r_estimate,sigma_estimate,r_error_percent,sigma_error_percent,pde_mse,pde_correlation,runtime_minutes
mean,0.050495,0.180115,0.989046,9.942599,0.003653,9.999992e-01,17.975854
std,0.000185,0.008283,0.369205,4.141409,0.001886,5.252344e-07,8.599055
min,0.050254,0.165210,0.508007,5.594114,0.000961,9.999983e-01,13.194445
max,0.050826,0.188812,1.652273,17.394923,0.006439,9.999997e-01,39.319564


Saved: 04_2_pinn_10_run_individual_summary.csv
